In [ ]:
# Jewish_Cemetery_Children_Soldiers_Match.py
# 8/14/2025

# Match Soldiers Using children of Jewish Cemeteries Parents

# fileInput=grave_source+"_FAG_Mereged"
# fileIn=DirMain+"Analysis\\"+fileInput+".csv"

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd


import json

import sys
import random
    
def normalize(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r'[^a-z\s]', '', s)  # keep only letters and spaces
    s = re.sub(r'\s+', ' ', s)      # collapse multiple spaces
    return s.strip()


def haystack_contains_needle(needle_df,haystack_df):
    mask1 = [
    normalize(parent) in normalize(spouse)
    for parent,spouse in zip(
        needle_df,
        haystack_df,  
    )
    ]
    return mask1

def  haystack_contains_needle_fuzzy(dataframe_df,needle,haystack):
    match=haystack+'_match' 
    dataframe_df[match] = dataframe_df.apply(lambda row: all_parts_in(row[needle], row[haystack]),axis=1)
    mask=dataframe_df[match]
    return dataframe_df, mask

def  search_dataframe(dataframe,column,search_text):
    dataframe_extract=dataframe[dataframe[column]==search_text]
    return dataframe_extract

def clean_dataframe(dataframe_df,column):
    delete_list=['Pvt ','PTE ','Dr. ','Rev ','MAJ ','GEN ','Mrs ','CPT ',',PFC ','SGT ','Col ','LT ','Dr ',\
                ' Sr',' Jr','\.','Minor ','Infant ','Boy ','Girl ','Twins ','PFC ','TEC5 ',\
                'PVT ','S Sgt ','Capt ','Pfc ','TSgt ','COL ','Tec5 ','TEC4 ','1','2','III','II','Lt','CPL','SSgt','Sgt',\
                'Staff Sergeant ','TEC 5 ','Maj ','GM3 ','TEC 4 ','Tec4 ','RM3 ','Corp ','T 5 ','MS ','  ']
    dataframe_df=dataframe_df.copy()
    for item in delete_list:
        dataframe_df[column]= dataframe_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return dataframe_df 

print("***Jewish_Cemetery_Children_Soldiers_Match--- Started *****")
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

fileInput="Jewish_Cemeteries_Children_Name_Graves_Combined"
fileABMCInput="ABMC_WWII_Soldiers"
fileOutput="Jewish_Cemeteries_Children_Name_Graves_Parsed"
fileInputSoD="star_david_20250915_df"

fileIn=DirMain+"Affiliation_Reverse_Children\\Inputs\\"+fileInput+".csv"
fileABMCIn=DirMain+"Affiliation_Reverse_Children\\Inputs\\"+fileABMCInput+".csv"
fileOut=DirMain+"Affiliation_Reverse_Children\\Outputs\\"+fileOutput+".csv"
fileInSoD=DirMain+"Affilation_Analysis\\Inputs\\"+fileInputSoD+".csv"

children_names_df=pd.read_csv(fileIn, encoding='ISO-8859-1')
soldiers_df=pd.read_csv(fileABMCIn, encoding='ISO-8859-1', on_bad_lines='skip')
star_david_df=pd.read_csv(fileInSoD)
star_david_df = star_david_df.reset_index().rename(columns={'Name_Index': 'sod_Name_Index'})

print(list(soldiers_df))
input('')


# ['lastName', 'firstName', 'middleName', 'fullName', 'nickName', 'birthDate', 'birthMonth', 'birthYear', 'deathDate', 'deathMonth', 'deathYear', 'birthCityName', 'birthStateName', 'birthCountryName', 'cemeteryName', 'countryName', 'cemeteryId', 'links', 'intermentHasPhoto', 'Parents', 'Spouses', 'Siblings', 'Children', 'prob_jewish']

 # ['war', 'Name_Index', 'Last_Name', 'First_Name', 'Middle_Name', 'Middle_Initial', 'Full_Name', 'Nick_Name', 'birthDate', 'birthMonth', 'birthYear', 'deathDate', 'Birth_index', 'Death_index', 'deathMonth', 'deathYear', 'birthCityName', 'birthStateName', 'birthCountryName', 'cemeteryName', 'countryName', 'cemeteryId', 'links', 'intermentHasPhoto', 'Parents', 'Spouses', 'Siblings', 'Children']
# soldiers_df_clean=clean_dataframe(soldiers_df,'Full_Name')
soldiers_df_clean=soldiers_df.copy()

matching_soldiers_children_df=pd.merge(
    soldiers_df_clean,
    children_names_df,
    left_on=['Last_Name'],
    right_on=['lastName'],    
    how='inner')

matched_soldiers_children_list=['war','Name_Index','Last_Name','First_Name','Middle_Name','Middle_Initial','Full_Name'\
                                ,'Nick_Name','birthDate_x','birthMonth_x','birthYear_x','deathDate_x','Birth_index','Death_index'\
                                ,'deathMonth_x','deathYear_x','birthCityName_x','birthStateName_x','birthCountryName_x','cemeteryName_x'\
                                ,'countryName_x','cemeteryId_x','links_x','intermentHasPhoto_x','Parents_x','Spouses_x','Siblings_x'\
                                ,'Children_x','lastName','firstName','middleName','fullName','nickName','birthDate_y','birthMonth_y'\
                                ,'birthYear_y','deathDate_y','deathMonth_y','deathYear_y','birthCityName_y','birthStateName_y'\
                                ,'birthCountryName_y','cemeteryName_y','countryName_y','cemeteryId_y','links_y','intermentHasPhoto_y'\
                                ,'Parents_y','Spouses_y','Siblings_y','Children_y','prob_jewish']


needle_df=matching_soldiers_children_df['Full_Name']
haystack_df=matching_soldiers_children_df['Children_y']
mask=haystack_contains_needle(needle_df,haystack_df)

matched_soldiers_children_df=matching_soldiers_children_df[mask]
matched_soldiers_children_df.reset_index(drop=True, inplace = True)

matched_soldiers_children_df=matched_soldiers_children_df.copy()
matched_soldiers_children_df['age_at_birth']=matched_soldiers_children_df['birthYear_x']-matched_soldiers_children_df['birthYear_y']

matched_soldiers_children_sod_df=pd.merge(
    matched_soldiers_children_df,
    star_david_df,
    left_on=[ 'Name_Index'],
    right_on=['sod_Name_Index'],
    how='left')


matched_soldiers_children_sod_df['Star_of_David'] = matched_soldiers_children_sod_df['sod_Name_Index'].fillna('False')
matched_soldiers_children_sod_df['Star_of_David']=matched_soldiers_children_sod_df['Star_of_David']!='False'
matched_soldiers_children_sod_df.drop_duplicates(inplace=True)

reorder_format=['Name_Index','Full_Name','Parents_x','fullName','Children_y','prob_jewish','Star_of_David','cemeteryName_y','cemeteryId_y'
                ,'age_at_birth','birthYear_x','birthStateName_x','birthCountryName_x','cemeteryName_x','birthYear_y'\
                ,'birthStateName_y','birthCountryName_y']
rename_format=['Name_Index','soldiers_full_name','soldiers_parents','partent_full_name','partent__children','prob_jewish','partent_cemetery_name','partent_cemetery_id'\
               ,'Star_of_David','age_at_birth','soldiers_birth_year','soldiers_birth_state','soldiers_birth_country','soldiers_cemetery_name'\
               ,'partent_birth_year','partent_birth_state','partent_birth_country']

matched_soldiers_children_sod_df=matched_soldiers_children_sod_df[reorder_format]
matched_soldiers_children_sod_df.columns=rename_format
# matched_soldiers_children_sod_df = matched_soldiers_children_sod_df.sort_values(by='name_index',ascending=True)
matched_soldiers_children_sod_df.drop_duplicates(inplace=True)
matched_soldiers_children_sod_df.reset_index(drop=True, inplace = True)



fileOutDf=DirMain+"Affiliation_Reverse_Children\\Outputs\\matched_soldiers_children_sod_df.csv"
matched_soldiers_children_sod_df.to_csv(fileOutDf,mode="w", index=False,header=True)
print("\n")
print('Uploaded To: ',fileOutDf,' • Number of Records: ',len(matched_soldiers_children_sod_df))
# input('')


print("***Generate_FAG_children_index--- Complete *****")


***Jewish_Cemetery_Children_Soldiers_Match--- Started *****
['war', 'Name_Index', 'Last_Name', 'First_Name', 'Middle_Name', 'Middle_Initial', 'Full_Name', 'Nick_Name', 'birthDate', 'birthMonth', 'birthYear', 'deathDate', 'Birth_index', 'Death_index', 'deathMonth', 'deathYear', 'birthCityName', 'birthStateName', 'birthCountryName', 'cemeteryName', 'countryName', 'cemeteryId', 'links', 'intermentHasPhoto', 'Parents', 'Spouses', 'Siblings', 'Children']


In [9]:
matched_soldiers_children_sod_df=pd.merge(
    matched_soldiers_children_df,
    star_david_df,
    left_on=[ 'Name_Index'],
    right_on=['sod_Name_Index'],
    how='left')
matched_soldiers_children_sod_df['Full_Name']

0       Philip Abrahamson
1         Benjamin Abrams
2         Benjamin Abrams
3         Benjamin Abrams
4         Benjamin Abrams
              ...        
3010    Martin M Zozofsky
3011    Martin M Zozofsky
3012      T Charles Zubov
3013       William Zucker
3014      Jacob Zuckerman
Name: Full_Name, Length: 3015, dtype: object

children_names_explode_df['length']

In [ ]:
children_names_df=children_names_df.copy()
children_names_df=children_names_df.dropna(subset=['Children'])
children_names_df.reset_index(drop=True, inplace = True)

children_explode_df_out=pd.DataFrame([])
children_names_df['Children']=children_names_df['Children'].str.strip()
children_names_df['Children']=children_names_df['Children'].str.strip()
children_names_df=clean_dataframe(children_names_df,'Children')
children_names_df['split']=children_names_df['Children'].str.split("*")
children_names_explode_df = children_names_df.explode('split')
children_names_explode_df['split']=children_names_explode_df['split'].str.strip()
children_names_explode_df['split2']=children_names_explode_df['split'].str.split(" ")
children_names_explode_df['length']=children_names_explode_df['split2'].apply(len)
children_names_explode_df=children_names_explode_df.drop(children_names_explode_df[children_names_explode_df['length']<2].index)
list_index_2=children_names_explode_df[children_names_explode_df['length']==2].index

list_index_2=children_names_explode_df[children_names_explode_df['length']>=2].index
list_index_3=children_names_explode_df[children_names_explode_df['length']>=3].index
children_names_explode_df.loc[list_index_2,'name_index_2']=children_names_explode_df['split2'].loc[list_index_2].str[-1]+'-'+children_names_explode_df['split2'].loc[list_index_2].str[0] 
children_names_explode_df.loc[list_index_3,'name_index_3']=children_names_explode_df['split2'].loc[list_index_3].str[-1]+'-'+children_names_explode_df['split2'].loc[list_index_3].str[0] +'-'+children_names_explode_df['split2'].loc[list_index_3].str[1] 



In [1]:
list(soldiers_df)

NameError: name 'soldiers_df' is not defined